In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import itertools

In [ ]:
n_seeds = 10
n_iters = 1_000

gamma_arr = [0.9, 0.99, 0.995]

# Summary
Plot final run of *combined* SPMD-MC and Q-learn.

Basically, this code takes `2026_04_11.ipynb` and merges it with Q-learn.

## SPMD MC-SPMD on GridWorld ~GridWorld-loop~

- SPMD: `2026_04_15/exp_2`
- Qlearn: `2026_04_16/exp_2`

In [ ]:
exp_date = "2026_04_15"; env_name = ["GridWorld-5", "GridWorld-10", "GridWorld-50"]
# exp_date = "2026_04_08"; env_name = ["Garnet-50", "Garnet-200", "Garnet-1000"]
n_exp = 27

In [ ]:
score_1 = np.zeros((n_exp, n_seeds, n_iters))
sample_1 = np.zeros((n_exp, n_seeds, n_iters))
length_1 = np.zeros((n_exp, n_seeds), dtype=int)
root = os.path.join("..", "logs")
for i in range(len(score_1)):
    for j in range(n_seeds):
        df = pd.read_csv(os.path.join(root, exp_date, "exp_2", "run_%d" % i, "seed=%d.csv" % j))
        vals = df['point value'] 
        # vals = df['true value']
        length_1[i,j] = df_len = len(vals)
        score_1[i,j,:df_len] = vals
        
        df = pd.read_csv(os.path.join(root, exp_date, "exp_2", "run_%d" % i, "mixing_seed=%d.csv" % j))
        sample_1[i,j,:df_len] = df['cum_samples'][:df_len]
        sample_1[i,j,:df_len] += df['cum_est_samples'][:df_len]

opt_1 = np.zeros((n_exp//3, n_seeds))
for i in range(len(opt_1)):
    for j in range(n_seeds):
        df = pd.read_csv(os.path.join(root, exp_date, "exp_1", "run_%d" % i, "seed=%d.csv" % j))
        opt_1[i,j] = df['average value'].iloc[-1]

Bucket the data.

In [ ]:
buckets = np.append(
    np.append(np.arange(1e4, step=1e2), np.arange(1e4, 1e5, step=1e3)), 
    np.append(
        np.append(np.arange(1e5,1e6, step=1e4), np.arange(1e6,1e7, step=1e5)),
        np.arange(1e7,1e8, step=1e5)
    ),
)

# Bucketed scores. 1st index: number of exp, 2nd: seeds, 3rd: buckets
clean_1 = np.zeros((len(score_1), n_seeds, len(buckets)))
# Which bucket is last
clean_len_1 = np.zeros((len(score_1), n_seeds))
last_idx = -np.inf
for i in range(clean_1.shape[0]):
    for j in range(n_seeds):
        clean_len_1[i,j] = length_1[i,j]
        for k,x in enumerate(buckets):
            # finds index i such that len_i < x <= len_(i+1)
            if length_1[i,j] == 0:
                idx = 1
            else:
                idx = np.argmax(x <= sample_1[i,j,:length_1[i,j]])
            if idx != -1:
                clean_1[i,j,k]
                score_1[i,j,idx]
                clean_1[i,j,k] = score_1[i,j,idx]
            if idx < last_idx:
                clean_len_1[i,j] = k
            last_idx = idx

Now we will get Qlearn.

In [ ]:
exp_date = "2026_04_16"
# exp_date = "2026_04_25"
n_exp = 9

In [ ]:
score_2 = np.zeros((n_exp, n_seeds, n_iters))
sample_2 = np.zeros((n_exp, n_seeds, n_iters))
length_2 = np.zeros((n_exp, n_seeds), dtype=int)
root = os.path.join("..", "logs")
for i in range(len(score_2)):
    for j in range(n_seeds):
        df = pd.read_csv(os.path.join(root, exp_date, "exp_2", "run_%d" % i, "seed=%d.csv" % j))
        vals = df['point value'] 
        
        # vals = df['true value']
        length_2[i,j] = df_len = len(vals)
        score_2[i,j,:df_len] = vals[:n_iters]
        
        df = pd.read_csv(os.path.join(root, exp_date, "exp_2", "run_%d" % i, "mixing_seed=%d.csv" % j))
        sample_2[i,j,:df_len] = df['cum_samples'][:n_iters]
        sample_2[i,j,:df_len] += df['cum_est_samples'][:n_iters]

Bucket.

In [ ]:
# Bucketed scores. 1st index: number of exp, 2nd: seeds, 3rd: buckets
clean_2 = np.zeros((len(score_2), n_seeds, len(buckets)))
# Which bucket is last
clean_len_2 = np.zeros((len(score_2), n_seeds))
last_idx = -np.inf
for i in range(clean_2.shape[0]):
    for j in range(n_seeds):
        clean_len_2[i,j] = length_2[i,j]
        for k,x in enumerate(buckets):
            # finds index i such that len_i < x <= len_(i+1)
            if length_2[i,j] == 0:
                idx = 1
            else:
                idx = np.argmax(x <= sample_2[i,j,:length_2[i,j]])
            if idx != -1:
                clean_2[i,j,k]
                score_2[i,j,idx]
                clean_2[i,j,k] = score_2[i,j,idx]
            if idx < last_idx:
                clean_len_2[i,j] = k
            last_idx = idx

Select the experiment.

In [ ]:
i_d = 2

In [ ]:
plt.style.use('ggplot')
_, ax = plt.subplots(figsize=(5,4))

label_arr = ["SPMD MC-Fixed", "SPMD MC-Est", "SPMD MC-Dyn", "Q-learn"]
color_arr = ["blue", "black", "red", "green"]
lss_arr   = ["dotted", "dashed", "solid", "dashdot"]
z = 2.228
x_max = 0

for k,i in enumerate(range(3*i_d,3*(1+i_d))):
    least_len_idx = int(np.min(clean_len_1[i,:]))
    xs = buckets[:least_len_idx]
    mean = np.mean(clean_1[i,:,:least_len_idx] - np.outer(opt_1[i//3], np.ones(least_len_idx)), axis=0) 
    sig  = np.std(clean_1[i,:,:least_len_idx], axis=0)
    ax.plot(xs, mean, label=label_arr[k], color=color_arr[k], linestyle=lss_arr[k])
    ax.fill_between(xs, mean - z*sig, mean+z*sig, color=color_arr[k], alpha=0.05)
    if k != 1 and len(xs) > 0:
        x_max = max(x_max, np.max(xs))

# q-learn
i = i_d
least_len_idx = int(np.min(clean_len_2[i,:]))
xs = buckets[:least_len_idx]
mean = np.mean(clean_2[i,:,:least_len_idx] - np.outer(opt_1[i_d], np.ones(least_len_idx)), axis=0) 
sig  = np.std(clean_2[i,:,:least_len_idx], axis=0)
ax.plot(xs, mean, label=label_arr[3], color=color_arr[3], linestyle=lss_arr[3])
ax.fill_between(xs, mean - z*sig, mean+z*sig, color=color_arr[3], alpha=0.05)

ax.legend(loc="upper right")
ax.set(
    title=env_name[i_d//3] + r" with $\gamma$=%s" % gamma_arr[i_d % 3],
    xlabel="Samples",
    ylabel=r"Average optimality gap",
    xlim=(0,x_max*1.05),
)

**NOTE**: One major issue is that the number of iteraitons do not align. We will re-run these experiments so that they better align (using the dynamic sample count as the baseline).